In [0]:
from pyspark.sql.functions import current_timestamp, input_file_name

source_path = "abfss://landing@storageaccount.dfs.core.windows.net/salesforce/accounts/"
checkpoint_path = "abfss://checkpoints@storageaccount.dfs.core.windows.net/autoloader/salesforce/accounts/"
schema_path = "abfss://schemas@storageaccount.dfs.core.windows.net/autoloader/salesforce/accounts/"
target_table = "bronze_dev.salesforce.account_raw"

df = (
    spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "json")
        .option("cloudFiles.schemaLocation", schema_path)
        .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
        .load(source_path)
        .withColumn("_ingestion_timestamp", current_timestamp())
        .withColumn("_source_file", input_file_name())
)

query = (
    df.writeStream
        .format("delta")
        .option("checkpointLocation", checkpoint_path)
        .option("mergeSchema", "true")
        .trigger(availableNow=True)
        .toTable(target_table)
)